# Port Data Catalog Updater - Version 2

In [ ]:
import pandas as pd

# ======================================================
# CONFIG — CHANGE THESE EACH YEAR
# ======================================================
REPORTING_YEAR = 2023

EXCEL_FILE = f"PortPerformance{REPORTING_YEAR}.xlsx"
HISTORICAL_FILE = "Port_Data_20251112.csv"
OUTPUT_FILE = f"Annual_Port_Statistics_{REPORTING_YEAR}_UPDATE.csv"
#PERCENT_FILE = f"PortPerformance{REPORTING_YEAR-1}.xlsx"
# ======================================================
# LOAD WORKBOOK + HISTORICAL DATA
# ======================================================
xls = pd.ExcelFile(EXCEL_FILE)
#xls_minus1 = pd.ExcelFile(PERCENT_FILE)
df_hist = pd.read_csv(HISTORICAL_FILE)
df_hist['Port ID'] = df_hist['Port ID'].str.replace(",","").astype(int)

df_hist['Percent Change'] = df_hist['Percent Change'].str.replace(",","").astype(float) 
df_hist['Percent Change'] = df_hist['Percent Change'].round(1) 
df_hist_year = df_hist[df_hist["Reporting Year"] == REPORTING_YEAR]
df_hist_year_minus_1 = df_hist[df_hist["Reporting Year"] == REPORTING_YEAR - 1]
#df_hist_year_minus_1['Port ID'] = df_hist_year_minus_1['Port ID'].str.replace(",","").astype(int)
df_hist_year_minus_1['Volume'] = df_hist_year_minus_1['Volume'].str.replace(",","").astype(float)
records = []

if REPORTING_YEAR == 2022:
    portXrefs = pd.read_csv("Port_Xrefs.csv")
    portXrefs=portXrefs.set_index("Known_Port_Name", drop=True)['Best_Match'].to_dict()
    df_container = pd.read_excel("WCSC 2022 Container traffic.xlsx", header=[0, 1, 2])
else:
    df_container = pd.read_excel("2023 Annual TEUs from US Army.xlsx", header=[0, 1, 2])
    portXrefs = pd.read_csv("Port_Xrefs.csv")
    portXrefs=portXrefs.set_index("Known_Port_Name", drop=True)['Best_Match'].to_dict()



## Helper Functions

### Get State

In [ ]:
tmp = xls.parse("Ports by ICST")
# df['State'] = df['PORT_NAME'].str.split(', ').str[-1]
# df.head()
# state_abbrev_to_name = {
#     'AL': 'Alabama', 'AK': 'Alaska', 'AZ': 'Arizona', 'AR': 'Arkansas',
#     'CA': 'California', 'CO': 'Colorado', 'CT': 'Connecticut', 'DE': 'Delaware',
#     'FL': 'Florida', 'GA': 'Georgia', 'HI': 'Hawaii', 'ID': 'Idaho',
#     'IL': 'Illinois', 'IN': 'Indiana', 'IA': 'Iowa', 'KS': 'Kansas',
#     'KY': 'Kentucky', 'LA': 'Louisiana', 'ME': 'Maine', 'MD': 'Maryland',
#     'MA': 'Massachusetts', 'MI': 'Michigan', 'MN': 'Minnesota', 'MS': 'Mississippi',
#     'MO': 'Missouri', 'MT': 'Montana', 'NE': 'Nebraska', 'NV': 'Nevada',
#     'NH': 'New Hampshire', 'NJ': 'New Jersey', 'NM': 'New Mexico', 'NY': 'New York',
#     'NC': 'North Carolina', 'ND': 'North Dakota', 'OH': 'Ohio', 'OK': 'Oklahoma',
#     'OR': 'Oregon', 'PA': 'Pennsylvania', 'RI': 'Rhode Island', 'SC': 'South Carolina',
#     'SD': 'South Dakota', 'TN': 'Tennessee', 'TX': 'Texas', 'UT': 'Utah',
#     'VT': 'Vermont', 'VA': 'Virginia', 'WA': 'Washington', 'WV': 'West Virginia',
#     'WI': 'Wisconsin', 'WY': 'Wyoming', 'DC': 'District of Columbia',
#     'PR': 'Puerto Rico', 'VI': 'Virgin Islands', 'GU': 'Guam', 'AS': 'American Samoa'
# }

# Usage
#df['State_Full'] = df['State'].map(state_abbrev_to_name)
tmp.head()


### Get Station List form Historical Data

In [ ]:
ports_hist = df_hist[['Port_Name','Port ID','State', 'Region']].drop_duplicates()
#ports_hist['Port ID'] = ports_hist['Port ID'].str.replace(",","")
ports_hist.dropna(inplace=True)
ports_hist.to_csv("ports_hist.csv", index=False)



## Total Tonage

In [ ]:
df = xls.parse("Top Ports")
print("DF ORIG SHAPE",df.shape)
## Get Port ID from the POrts by ICST sheet
tmp = xls.parse("Ports by ICST")
tmp = tmp[["PORT","PORT_NAME"]].drop_duplicates()
df=pd.merge(df,tmp,how="left",left_on="PORT NAME",right_on="PORT_NAME")
df.drop(columns=["PORT_NAME"],inplace=True)
print("DF NEW SHAPE",df.shape)
# df['PORT'] = df['PORT'].astype(int) 

mapr = {'GRAND TOTAL':'TOTAL','FOREIGN TOTAL':'FOREIGN','IMPORTS':'IMPORTS','EXPORTS':'EXPORTS','DOMESTIC':'DOMESTIC'}
all_records = []
for idx,row in ports_hist.iterrows():
  #  print(row)
    port_id = row['Port ID']
    port_name = row['Port_Name'].strip()

    hld=df.loc[(df['PORT'] == port_id) | (df['PORT NAME'].str.strip() == port_name) ]
    hst = df_hist_year_minus_1.loc[(df_hist_year_minus_1['Port ID'] == port_id) & (df_hist_year_minus_1['Cargo Type'] == 'TOTAL TONNAGE')]
    
    if hld.shape[0]==0:
        print("No data for port ID:", port_id, row['Port_Name'])
        continue
    for tt in ['GRAND TOTAL','FOREIGN TOTAL','IMPORTS','EXPORTS','DOMESTIC']:
        tp = mapr[tt]
        vol_minus_1 = hst[hst['Trade Type'] == tp]['Volume'].values[0] if not hst.empty else None              
       
        if vol_minus_1:
            percent_change = ((hld[tt].values[0] - vol_minus_1) / vol_minus_1) * 100
            percent_change = round(percent_change, 1)
        else:
            percent_change = 0
         #   print(f"No historical volume for port ID {row['Port_Name']} {port_id}, trade type {tp}. Cannot calculate percent change.")




        tmp = {
          'Cargo Type': 'TOTAL TONNAGE', 
          'Port ID': int(port_id), 
          'Port_Name': row['Port_Name'], 
          'Region': row['Region'],
          'Reporting Year': REPORTING_YEAR,
          'State': row['State'], 
          'Trade Type':tp, 
          'Units':'Short Tons', 
          'Port Ranking': hld['RANK'].values[0],
          'Percent Change':percent_change,
          'Volume': hld[tt].values[0]
        }
       
        all_records.append(tmp)

totalTonnage = pd.DataFrame(all_records)
print("Total Tonnage shape:",totalTonnage.shape)


        

### Total Check

#### Internal Checks

In [ ]:
nbad=0
ngood=0
ntotal=0

for pid in totalTonnage['Port ID'].unique():
    row = totalTonnage[totalTonnage['Port ID'] == pid]
    hld={}
    bad=False
    ntotal+=1
    for tt in row['Trade Type'].unique():
        hld[tt] = row[row['Trade Type'] == tt]['Volume'].values[0]
    frgn = hld['FOREIGN']
    dmst = hld['DOMESTIC']
    tot = hld['TOTAL']
    impt = hld['IMPORTS']
    expo = hld['EXPORTS']
    if frgn != impt + expo:
        print(f"Mismatch for Port ID {pid}: FOREIGN {frgn} vs IMPORTS+EXPORTS {impt+expo} IMPT {impt} EXPO {expo}")
        bad=True
    if tot != dmst + frgn:
        print(f"Mismatch for Port ID {pid}: TOTAL {tot} vs DOMESTIC+FOREIGN {dmst+frgn} DMST {dmst} FRGN {frgn}")
        bad=True

    if bad:
        nbad+=1
    else:
        ngood+=1

print(f"Checked {ntotal} ports with {nbad} mismatches and {ngood} good records.")

# Get value counts for the 'Trade Type' column
counts = totalTonnage['Trade Type'].value_counts()

# Check if all counts are equal
all_equal = counts.nunique() == 1
print(f"All types have equal counts: {all_equal}")

# See which types have different counts
print(f"\nMin count: {counts.min()}")
print(f"Max count: {counts.max()}")
print(f"Difference: {counts.max() - counts.min()}")
#    print(f"Port ID {pid}, Total Volume from Trade Types: {tot}, Total Volume from TOTAL row: {row[row['Trade Type'] == 'TOTAL']['Volume'].values[0]}")

#### Historical Check

In [ ]:
tot_hist = df_hist_year.loc[df_hist_year['Cargo Type'] == "TOTAL TONNAGE"].copy()
tot_hist.shape
#tot_hist['Port ID']= tot_hist['Port ID'].str.replace(",","")
tot_hist['Volume']= tot_hist['Volume'].str.replace(",","").astype(int)

tot_hist['Port Ranking']= tot_hist['Port Ranking'].astype(int)

totalTonnage.sort_values(by=["Port ID","Trade Type"],inplace=True)
totalTonnage.reset_index(drop=True,inplace=True)
tot_hist.sort_values(by=["Port ID","Trade Type"],inplace=True)
tot_hist.reset_index(drop=True,inplace=True)
nchecks=0
nbad=0
for idx,row in totalTonnage.iterrows():
    prt=row['Port ID']
    tp=row['Trade Type'] 
    nchecks+=1
    print(f"Checking Port ID {prt}, Port Name {row['Port_Name']}, Trade Type {tp}..." )
    hst= tot_hist.loc[(tot_hist['Port ID']==prt) & (tot_hist['Trade Type']==tp)] 
    print(f"Historical record found: {not hst.empty}")     
    for ii in row.index:
        if str(row[ii]).strip() != str(hst[ii].values[0]).strip() :
             print(f"Mismatch for Port ID {prt}, Trade Type {tp}, Field {ii}: New={row[ii]} vs Hist={hst[ii].values[0]}")
             nbad+=1

print(f"Completed {nchecks} checks with {nbad} mismatches.")
totalTonnage_Stns = set(totalTonnage['Port_Name'].unique().tolist())
totalTonnage_hist_Stns = set(tot_hist['Port_Name'].unique().tolist())


print(f"Completed {nchecks} checks with {nbad} mismatches.")
print("Update Shape",totalTonnage.shape)
print("Update Shape",tot_hist.shape)
print("Total Tonnage Ports NEW not in HIST:", totalTonnage_Stns - totalTonnage_hist_Stns)
print("Total Tonnage Ports HIST not in NEW:", totalTonnage_hist_Stns - totalTonnage_Stns)

## Dry Bulk

In [ ]:

df = xls.parse("Dry Bulk")
df.columns = df.columns.astype(str).str.strip().str.upper()
print("DRY BULK TONNAGE COLUMNS:", df.columns.tolist())
df['PORT'] = df['PORT'].astype('Int64')
## Get Port ID from the POrts by ICST sheet
# tmp = xls.parse("Ports by ICST")
# tmp = tmp[["PORT","PORT_NAME"]].drop_duplicates()
# df=pd.merge(df,tmp,how="left",left_on="PORT NAME",right_on="PORT_NAME")
# df.drop(columns=["PORT_NAME"],inplace=True)
# print("DF NEW SHAPE",df.shape)

all_records = []
for idx,row in ports_hist.iterrows():
  #  print(row)
    port_id = int(row['Port ID'])
    port_name = row['Port_Name'].strip()
    hld=df.loc[(df['PORT'] == port_id) | (df['PORT NAME'].str.strip() == port_name)]
    if hld.shape[0]==0:
        print("No data for port ID:",port_id,row['Port_Name'])
        continue
    
    hst = df_hist_year_minus_1.loc[(df_hist_year_minus_1['Port ID'] == port_id) & (df_hist_year_minus_1['Cargo Type'] == 'DRY BULK')]

    for tt in ['TOTAL','FOREIGN','IMPORTS','EXPORTS','DOMESTIC']:
        volume=int(hld[tt].values[0]) if pd.notna(hld[tt].values[0]) else 0
        vol_minus_1 = hst[hst['Trade Type'] == tt]['Volume'].values[0] if not hst.empty else None              
       
        if vol_minus_1:
            percent_change = ((hld[tt].values[0] - vol_minus_1) / vol_minus_1) * 100
            percent_change = round(percent_change, 1)
        else:
            percent_change = 0
            print(f"No historical volume for port ID {row['Port_Name']} {port_id}, trade type {tt}. Cannot calculate percent change.")




        tmp = {
          'Cargo Type': 'DRY BULK', 
          'Port ID': port_id, 
          'Port_Name': row['Port_Name'], 
          'Region': row['Region'],
          'Reporting Year': REPORTING_YEAR,
          'State': row['State'], 
          'Trade Type':tt, 
          'Units':'Short Tons', 
          'Port Ranking': hld.index.values[0]+1,
          'Percent Change':percent_change,
          'Volume': volume
        }
       
        all_records.append(tmp)

dryBulk = pd.DataFrame(all_records)
print("Dry Bulk shape:",dryBulk.shape)

        

### Dry Bulk Check

#### Internal Check

In [ ]:
nbad=0
ngood=0
ntotal=0

for pid in dryBulk['Port ID'].unique():
    row = dryBulk[dryBulk['Port ID'] == pid]
    hld={}
    bad=False
    ntotal+=1
    for tt in row['Trade Type'].unique():
        hld[tt] = row[row['Trade Type'] == tt]['Volume'].values[0]
    frgn = hld['FOREIGN']
    dmst = hld['DOMESTIC']
    tot = hld['TOTAL']
    impt = hld['IMPORTS']
    expo = hld['EXPORTS']
    if frgn != impt + expo:
        print(f"Mismatch for Port ID {pid}: FOREIGN {frgn} vs IMPORTS+EXPORTS {impt+expo} IMPT {impt} EXPO {expo}")
        bad=True
    if tot != dmst + frgn:
        print(f"Mismatch for Port ID {pid}: TOTAL {tot} vs DOMESTIC+FOREIGN {dmst+frgn} DMST {dmst} FRGN {frgn}")
        bad=True

    if bad:
        nbad+=1
    else:
        ngood+=1

print(f"Checked {ntotal} ports with {nbad} mismatches and {ngood} good records.")

# Get value counts for the 'Trade Type' column
counts = dryBulk['Trade Type'].value_counts()

# Check if all counts are equal
all_equal = counts.nunique() == 1
print(f"All types have equal counts: {all_equal}")

# See which types have different counts
print(f"\nMin count: {counts.min()}")
print(f"Max count: {counts.max()}")
print(f"Difference: {counts.max() - counts.min()}")
#    print(f"Port ID {pid}, Total Volume from Trade Types: {tot}, Total Volume from TOTAL row: {row[row['Trade Type'] == 'TOTAL']['Volume'].values[0]}")

#### Historical Check

In [ ]:
dryBulk_hist = df_hist_year.loc[df_hist_year['Cargo Type'] == "DRY BULK"].copy()
dryBulk_hist.shape
#dryBulk_hist['Port ID']= dryBulk_hist['Port ID'].astype(str).str.replace(",","")
dryBulk_hist['Volume']= dryBulk_hist['Volume'].astype(str).str.replace(",","").str.replace("nan","0").astype(int)

dryBulk_hist['Port Ranking']= dryBulk_hist['Port Ranking'].astype(int)

nchecks=0
nbad=0
for idx,row in dryBulk.iterrows():
    prt=row['Port ID']
    tp=row['Trade Type'] 
    nchecks+=1
    hst= dryBulk_hist.loc[(dryBulk_hist['Port ID']==prt) & (dryBulk_hist['Trade Type']==tp)]      
    for ii in row.index:
        if str(row[ii]).strip() != str(hst[ii].values[0]).strip() and str(ii) != "Percent Change":
             print(f"Mismatch for Port ID {prt}, Trade Type {tp}, Field {ii}: New={row[ii]} vs Hist={hst[ii].values[0]}")
             nbad+=1

dryBulk_Stns = set(dryBulk['Port_Name'].unique().tolist())
dryBulk_hist_Stns = set(dryBulk['Port_Name'].unique().tolist())



print(f"Completed {nchecks} checks with {nbad} mismatches.")
print("Update Shape",dryBulk.shape)
print("Update Shape",dryBulk_hist.shape)
print("Dry Bulk Ports NEW not in HIST:", dryBulk_Stns - dryBulk_hist_Stns)
print("Dry Bulk Ports HIST not in NEW:", dryBulk_hist_Stns - dryBulk_Stns)

## Vessel Calls

In [ ]:
df = xls.parse("Ports by ICST")

all_records = []

for idx,row in ports_hist.iterrows():
    port_id = int(row['Port ID'])
    port_name = row['Port_Name'].strip()
 #   tmp = df.loc[(df['PORT'] == port_id) ]
 #   print("Processing Port ID:", port_id, row['Port_Name'],tmp.shape)
    if df.loc[(df['PORT'] == port_id) | (df['PORT_NAME'].str.strip() == port_name)].shape[0]==0:
        print("No data for port ID:",port_id,row['Port_Name'])
        continue
    hst = df_hist_year_minus_1.loc[(df_hist_year_minus_1['Port ID'] == port_id) & (df_hist_year_minus_1['Cargo Type'] == 'VESSEL CALLS')]
    
    nmiss=0
    for tt in ["Container","Other Freight Barge","Dry Bulk","Dry Bulk Barge","Other Freight"]:

        tmp = df.loc[((df['PORT'] == port_id) | (df['PORT_NAME'].str.strip() == port_name)) & (df['CATEGORY'].str.strip() == tt)]
        vol_minus_1 = hst[hst['Trade Type'] == tt]['Volume'].values[0] if not hst.empty else None              
       
        
        if tmp.shape[0] > 0:
            vol_in = tmp['INBOUND'].values[0]
            vol_out = tmp['OUTBOUND'].values[0]
            vol_mean = (vol_in + vol_out) / 2
            
        else:
            print("MIS ",port_id,tt,tmp.shape)
            vol_mean=0
            nmiss+=1

        if vol_minus_1 and vol_minus_1 > 0:
            val = tmp.loc[tmp['CATEGORY'].str.strip() == tt, 'INBOUND'].values[0] if not tmp.empty else 0   
            percent_change = ((vol_mean - vol_minus_1) / vol_minus_1) * 100
            percent_change = round(percent_change, 1)
        else:
            percent_change = 0
            print(f"No historical volume for port ID {row['Port_Name']} {port_id}, trade type {tt}. Cannot calculate percent change.")

        hld = {
            'Cargo Type': 'VESSEL CALLS', 
            'Port ID': port_id, 
            'Port_Name': row['Port_Name'], 
            'Region': row['Region'],
            'Reporting Year': REPORTING_YEAR,
            'State': row['State'], 
            'Trade Type':tt, 
            'Units':'Vessel Calls', 
            'Port Ranking': None,
            'Percent Change':percent_change,
            'Volume': vol_mean
            }
        all_records.append(hld)
    if nmiss == 5:
        print("No data for Port ID:", port_id, row['Port_Name'])
vesselCalls = pd.DataFrame(all_records)
print("Vessel Calls shape:",vesselCalls.shape)



### Vessel Calls Check

In [ ]:
vc_hist = df_hist_year.loc[df_hist_year['Cargo Type'] == "VESSEL CALLS"].copy()

print(vc_hist.shape)
vc_hist['Port ID']= vc_hist['Port ID'].astype(str).str.replace(",","").astype(int)
vc_hist['Volume']= vc_hist['Volume'].astype(str).str.replace(",","").str.replace("nan","0").astype(float)

B=vesselCalls.copy()
A=vc_hist.copy()
nchecks=0
nbad=0
for idx,row in A.iterrows():
    prt=row['Port ID']
    tp=row['Trade Type'].strip()
    nchecks+=1
    hst= B.loc[(B['Port ID']==prt) & (B['Trade Type'].str.strip()==tp)] 
    if hst.shape[0] > 1:
        print("Multiple HIST records for Port ID:",prt," Trade Type:",tp,row)
    elif hst.shape[0] == 0:
        print("No mmm NEW data for Port ID:", prt, row['Port_Name'], " Trade Type:", tp)
        continue

    if len(row)==0:
        print("No NEW data for Port ID:", prt, " Trade Type:", tp)
        continue
    for ii in row.index:
        if str(row[ii]).strip() != str(hst[ii].values[0]).strip() and str(ii) != "Port Ranking":
             print(f"Mismatch for Port ID {prt}, Trade Type {tp}, Field {ii}: Old={row[ii]} vs New={hst[ii].values[0]}")
             nbad+=1

vesselCalls_Stns = set(vesselCalls['Port_Name'].unique().tolist())
vc_hist_Stns = set(vc_hist['Port_Name'].unique().tolist())



print(f"Completed {nchecks} checks with {nbad} mismatches.")
print("Update Shape",vesselCalls.shape)
print("  Hist Shape",vc_hist.shape)
print("Vessel Calls Ports NEW not in HIST:", vesselCalls_Stns - vc_hist_Stns)
print("Vessel Calls Ports HIST not in NEW:", vc_hist_Stns - vesselCalls_Stns)

## Top 5 Commodities

In [ ]:
## Fix this to get commodities out of 2021 port performance list


df = xls.parse("Ports by Commodity")

all_records = []

for _,row in ports_hist.iterrows():
    port_id = int(row['Port ID'])
    port_name = row['Port_Name'].strip()

    hld=df.loc[(df['PORT'] == port_id) | (df['Port Name'].str.strip() == port_name)]
    total = hld['TOTAL'].sum()
    hld=hld.sort_values(by="TOTAL", ascending=False).reset_index(drop=True).iloc[:5]
    for idx,row2 in hld.iterrows():
        tt=row2['Commodity Name'].strip() 
        volume=row2['TOTAL']
        hst = df_hist_year_minus_1.loc[(df_hist_year_minus_1['Port ID'] == port_id) & (df_hist_year_minus_1['Cargo Type'] == 'TOP 5 COMMODITIES') & (df_hist_year_minus_1['Trade Type'] == tt)]
        vol_minus_1 = hst['Volume'].values[0] if not hst.empty else 0
        if vol_minus_1 and vol_minus_1 > 0:
            val = row2['TOTAL']   
            percent_change = ((val - vol_minus_1) / vol_minus_1) * 100
            percent_change = round(percent_change, 1)
        else:
            percent_change = 0
            print(f"No historical volume for port ID {row['Port_Name']} {port_id}, trade type {tt}. Cannot calculate percent change.")




        tmp = {
            'Cargo Type': 'TOP 5 COMMODITIES', 
            'Port ID': port_id, 
            'Port_Name': row['Port_Name'], 
            'Region': row['Region'],
            'Reporting Year': REPORTING_YEAR,
            'State': row['State'], 
            'Trade Type':tt, 
            'Units':'Short Tons', 
            'Port Ranking': None,
            'Percent Change':percent_change,
            'Volume': volume
            }
        if volume> 0:
           all_records.append(tmp)
## Add Total
    hst = df_hist_year_minus_1.loc[(df_hist_year_minus_1['Port ID'] == port_id) & (df_hist_year_minus_1['Cargo Type'] == 'TOP 5 COMMODITIES') & (df_hist_year_minus_1['Trade Type'] == 'TOTAL')]
    vol_minus_1 = hst['Volume'].values[0] if not hst.empty else 0
    if vol_minus_1 and vol_minus_1 > 0:
        val = total   
        percent_change = ((val - vol_minus_1) / vol_minus_1) * 100
        percent_change = round(percent_change, 1)
    else:
        percent_change = 0
        print(f"No historical volume for port ID {row['Port_Name']} {port_id}, trade type {tt}. Cannot calculate percent change.")

    tmp = {
            'Cargo Type': 'TOP 5 COMMODITIES', 
            'Port ID': port_id, 
            'Port_Name': row['Port_Name'], 
            'Region': row['Region'],
            'Reporting Year': REPORTING_YEAR,
            'State': row['State'], 
            'Trade Type':'TOTAL', 
            'Units':'Short Tons', 
            'Port Ranking': None,
            'Percent Change':None,
            'Volume': total
            }
    if total > 0:
       all_records.append(tmp)

top5Com = pd.DataFrame(all_records)

print("Top 5 Commodities shape:",top5Com.shape)


    
    

### Top 5 Commodity Check

In [ ]:
top5_hist=df_hist_year.loc[df_hist_year['Cargo Type'] == 'TOP 5 COMMODITIES']
top5_hist.head()
#top5_hist["Port ID"] = top5_hist["Port ID"].str.replace(",","").astype(int)
top5_hist["Volume"] = top5_hist["Volume"].str.replace(",","").astype(int)
B=top5Com.copy()
A=top5_hist.copy()
nchecks=0
nbad=0
for idx,row in A.iterrows():
    prt=row['Port ID']
    tp=row['Trade Type'].strip()
    nchecks+=1
    hst= B.loc[(B['Port ID']==prt) & (B['Trade Type'].str.strip()==tp)] 
    if hst.shape[0] > 1:
        print("Multiple HIST records for Port ID:",prt," Trade Type:",tp,row)
    elif hst.shape[0] == 0:
        print("No mmm NEW data for Port ID:", prt, row['Port_Name'], " Trade Type:", tp)
        continue

    if len(row)==0:
        print("No NEW data for Port ID:", prt, " Trade Type:", tp)
        continue
    for ii in row.index:
        if is_number(row[ii]) and is_number(hst[ii].values[0]):
            if abs(float(row[ii]) - float(hst[ii].values[0])) > 0.0 and str(ii) != "Port Ranking":
                print(f"Numeric Mismatch for Port ID {prt}, Trade Type {tp}, Field {ii}: New={row[ii]} vs Hist={hst[ii].values[0]}  ")
                nbad+=1
                if ii == 'Volume':
                    badsn[(prt,tp)] = (row[ii], hst[ii].values[0])
        elif str(row[ii]).strip() != str(hst[ii].values[0]).strip()  and str(ii) != "Port Ranking":
             print(f"Mismatch for Port ID {prt}, Trade Type {tp}, Field {ii}: Hist={row[ii]} vs Row={hst[ii].values[0]}")
             nbad+=1
        

top5Com_Stns = set(top5Com['Port_Name'].unique().tolist())
top5_hist_Stns = set(top5_hist['Port_Name'].unique().tolist())



print(f"Completed {nchecks} checks with {nbad} mismatches.")
print("Update Shape",top5Com.shape)
print("  Hist Shape",top5_hist.shape)
print("Vessel Calls Ports NEW not in HIST:", top5Com_Stns - top5_hist_Stns)
print("Vessel Calls Ports HIST not in NEW:", top5_hist_Stns - top5Com_Stns)

## Top 5 Farm/Agriculture Commodities

In [ ]:
top5Ag_hist = df_hist.loc[df_hist['Cargo Type'] == 'TOP 5 FOOD/FARM COMMODITIES']
ag_commodities_hist = top5Ag_hist['Trade Type'].unique().tolist()
df = xls.parse("Ports by Commodity")
all_records = []

for _,row in ports_hist.iterrows():
    port_id = int(row['Port ID'])
    port_name = row['Port_Name'].strip()
 #   hld=top5Ag.loc[top5Ag['PORT'] == port_id]
    total = df.loc[((df['PORT'] == port_id) | (df['Port Name'].str.strip() == port_name)) & (df['Commodity Group'] >= 6000) & (df['Commodity Group'] < 7000),'TOTAL'].sum()  
    hld = df.loc[((df['PORT'] == port_id) | (df['Port Name'].str.strip() == port_name)) & (df['Commodity Group'] >= 6000) & (df['Commodity Group'] < 7000)]
    hld=hld.sort_values(by="TOTAL", ascending=False).reset_index(drop=True).iloc[:5]
    for idx,row2 in hld.iterrows():
        tt=row2['Commodity Name'].strip() 
        volume=row2['TOTAL']
        hst = df_hist_year_minus_1.loc[(df_hist_year_minus_1['Port ID'] == port_id) & (df_hist_year_minus_1['Cargo Type'] == 'TOP 5 FOOD/FARM COMMODITIES') & (df_hist_year_minus_1['Trade Type'] == tt)]
        vol_minus_1 = hst['Volume'].values[0] if not hst.empty else 0
        if vol_minus_1 and vol_minus_1 > 0:
            val = row2['TOTAL']   
            percent_change = ((val - vol_minus_1) / vol_minus_1) * 100
            percent_change = round(percent_change, 1)
        else:
            percent_change = 0
            print(f"No historical volume for port ID {row['Port_Name']} {port_id}, trade type {tt}. Cannot calculate percent change.")


        tmp = {
            'Cargo Type': 'TOP 5 FOOD/FARM COMMODITIES', 
            'Port ID': port_id, 
            'Port_Name': row['Port_Name'], 
            'Region': row['Region'],
            'Reporting Year': REPORTING_YEAR,
            'State': row['State'], 
            'Trade Type':tt, 
            'Units':'Short Tons', 
            'Port Ranking': None,
            'Percent Change':percent_change,
            'Volume': volume
            }
        if volume> 0:
           all_records.append(tmp)
## Add Total
    hst = df_hist_year_minus_1.loc[(df_hist_year_minus_1['Port ID'] == port_id) & (df_hist_year_minus_1['Cargo Type'] == 'TOP 5 FOOD/FARM COMMODITIES') & (df_hist_year_minus_1['Trade Type'] == 'TOTAL')]
    vol_minus_1 = hst['Volume'].values[0] if not hst.empty else 0
    if vol_minus_1 and vol_minus_1 > 0:
        val = total   
        percent_change = ((val - vol_minus_1) / vol_minus_1) * 100
        percent_change = round(percent_change, 1)
    else:
        percent_change = 0
        print(f"No historical volume for port ID {row['Port_Name']} {port_id}, trade type {tt}. Cannot calculate percent change.")



    tmp = {
            'Cargo Type': 'TOP 5 FOOD/FARM COMMODITIES', 
            'Port ID': port_id, 
            'Port_Name': row['Port_Name'], 
            'Region': row['Region'],
            'Reporting Year': REPORTING_YEAR,
            'State': row['State'], 
            'Trade Type':'TOTAL', 
            'Units':'Short Tons', 
            'Port Ranking': None,
            'Percent Change':percent_change,
            'Volume': total
            }
    if total > 0:
       all_records.append(tmp)

top5Ag = pd.DataFrame(all_records)

print("Top 5 Farm/Food Commodities shape:",top5Ag.shape)




### Top 5 Farm/Agriculture Check

In [ ]:
from pandas.api.types import is_number
top5Ag_hist = df_hist_year.loc[df_hist_year['Cargo Type'] == 'TOP 5 FOOD/FARM COMMODITIES']
#ag_commodities_hist = top5Ag_hist['Trade Type'].unique().tolist()


#top5Ag_hist["Port ID"] = top5Ag_hist["Port ID"].str.replace(",","").astype(int)
top5Ag_hist["Volume"] = top5Ag_hist["Volume"].str.replace(",","").astype(int)
B=top5Ag.copy()
A=top5Ag_hist.copy()
nchecks=0
nbad=0
badsn={}
for idx,row in A.iterrows():
    prt=row['Port ID']
    tp=row['Trade Type'].strip()
    nchecks+=1
    hst= B.loc[(B['Port ID']==prt) & (B['Trade Type'].str.strip()==tp)] 
    if hst.shape[0] > 1:
        print("Multiple HIST records for Port ID:",prt," Trade Type:",tp,row)
    elif hst.shape[0] == 0:
        print("No mmm NEW data for Port ID:", prt, row['Port_Name'], " Trade Type:", tp)
        continue

    if len(row)==0:
        print("No NEW data for Port ID:", prt, " Trade Type:", tp)
        continue
    for ii in row.index:
        
        if is_number(row[ii]) and is_number(hst[ii].values[0]):
            if abs(float(row[ii]) - float(hst[ii].values[0])) > 0.0 and str(ii) != "Port Ranking":
                print(f"Numeric Mismatch for Port ID {prt}, Trade Type {tp}, Field {ii}: New={row[ii]} vs Hist={hst[ii].values[0]}  ")
                nbad+=1
                if ii == 'Volume':
                    badsn[(prt,tp)] = (row[ii], hst[ii].values[0])
        elif str(row[ii]).strip() != str(hst[ii].values[0]).strip() and str(ii) != "Port Ranking":
       #      print(f"Mismatch for Port ID {prt}, Trade Type {tp}, Field {ii}: New={row[ii]} vs Hist={hst[ii].values[0]}  diff (H-U) {row[ii] - hst[ii].values[0]}")
             nbad+=1
             if ii == 'Volume':
                 badsn[(prt,tp)] = (row[ii], hst[ii].values[0])
        

top5Ag_Stns = set(top5Ag['Port_Name'].unique().tolist())
top5Ag_hist_Stns = set(top5Ag_hist['Port_Name'].unique().tolist())



print(f"Completed {nchecks} checks with {nbad} mismatches.")
print("Update Shape",top5Ag.shape)
print("  Hist Shape",top5Ag_hist.shape)
print("Vessel Calls Ports NEW not in HIST:", top5Ag_Stns - top5Ag_hist_Stns)
print("Vessel Calls Ports HIST not in NEW:", top5Ag_hist_Stns - top5Ag_Stns)


In [ ]:
hn={}
for k,v in badsn.items():
 #   print(f"Port ID {k[0]}, Trade Type {k[1]}: New={v[0]} vs Hist={v[1]}  diff (H-U) {v[0]-v[1]}")
    hn[k[0]] = v

ho={}
for k,v in badsn.items():
 #   print(f"Port ID {k[0]}, Trade Type {k[1]}: New={v[0]} vs Hist={v[1]}  diff (H-U) {v[0]-v[1]}")
    ho[k[0]] = v

for p,v in ho.items():
    if p in hn:
         print(f"Port ID {p}: OLD={v[0]} Org={v[1]} ")
         print(f"Port ID {p}: New={hn[p][0]} Org={hn[p][1]}  Dif=Org={hn[p][0] - hn[p][1]}\n")
    

## Container Activity

In [34]:
df = df_container.copy()
a=df.columns.tolist()
colsNew = []
for b in a:
    final=""

    for c in b:
        if "Unnamed:" in c:
            continue
        else:
            final += c.strip() + " "
    colsNew.append(final.strip())
    
df.columns = colsNew


df['IMPORTS'] = df['DOMESTIC InBound Loaded']+ df['FOREIGN InBound Loaded'] 
df['EXPORTS'] = df['DOMESTIC OutBound Loaded']+ df['FOREIGN OutBound Loaded'] 
# df['IMPORTS'] = df['FOREIGN InBound Loaded'] 
# df['EXPORTS'] = df['FOREIGN OutBound Loaded'] 
df['EMPTY'] = df['DOMESTIC InBound Empty'] + df['DOMESTIC OutBound Empty'] 
#df['TOTAL'] = df['Grand Total Loaded']
df['TOTAL'] = df['IMPORTS'] + df['EXPORTS'] + df['EMPTY']

df_contain = df[[ 'PORT NAME', 'STATE','TOTAL', 'IMPORTS','EXPORTS', 'EMPTY']].copy()
if REPORTING_YEAR == 2023:
    df_contain['PORT CODE'] = df['PORT CODE'].astype(int)     
else:
    df_contain['PORT CODE'] = None


In [35]:
ranks=df_contain[['PORT CODE','PORT NAME','TOTAL']].sort_values(by='TOTAL', ascending=False).reset_index(drop=True)

all_records = []
for _,row in ports_hist.iterrows():
    pid = row['Port ID']
    pname = row['Port_Name'].strip()
    xref = portXrefs[pname]
    hld = df_contain.loc[(df_contain['PORT CODE'] == pid) | (df_contain['PORT NAME'].str.strip() == xref)  ]
    rank = ranks.loc[(ranks['PORT CODE'] == pid) | (ranks['PORT NAME'].str.strip() == xref)].index.values[0] + 1 if not ranks.loc[(ranks['PORT CODE'] == pid) | (ranks['PORT NAME'].str.strip() == xref)].empty else None
    if hld.shape[0] > 0:
         for tt in ['TOTAL','IMPORTS','EXPORTS','EMPTY']:
            volume = hld[tt].values[0] if not hld.empty else 0
            hst = df_hist_year_minus_1.loc[(df_hist_year_minus_1['Port ID'] == pid) & (df_hist_year_minus_1['Cargo Type'] == 'CONTAINER') & (df_hist_year_minus_1['Trade Type'] == tt)]
            vol_minus_1 = hst['Volume'].values[0] if not hst.empty else 0
            if vol_minus_1 and vol_minus_1 > 0:
                val = hld[tt].values[0]   
                percent_change = ((val - vol_minus_1) / vol_minus_1) * 100
                percent_change = round(percent_change, 1)
            else:
                percent_change = 0
             #   print(f"No historical volume for port ID {row['Port_Name']} {pid}, trade type {tt}. Cannot calculate percent change.")
            tmp = {
                    'Cargo Type': 'CONTAINER', 
                    'Port ID': pid, 
                    'Port_Name': pname, 
                    'Region': row['Region'],
                    'Reporting Year': REPORTING_YEAR,
                    'State': row['State'], 
                    'Trade Type':tt, 
                    'Units':'Container TEUs', 
                    'Port Ranking': rank,
                    'Percent Change':percent_change,
                    'Volume': volume
            }
            all_records.append(tmp)
                
    else:
        print("No data for Port ID:", pid, row['Port_Name'])

container = pd.DataFrame(all_records)
print("Container shape:",container.shape)



No data for Port ID: 2338 Cincinnati-Northern KY, Ports of
No data for Port ID: 2348 Huntington-Tristate, KY, OH, WV
No data for Port ID: 4626 Kalama, WA Port of
No data for Port ID: 2306 Mid-America Port Commission
No data for Port ID: 2351 New Bourbon Port, MO
No data for Port ID: 3743 Northern Indiana, IN
No data for Port ID: 2358 Pittsburgh, PA Port of
No data for Port ID: 2255 Plaquemines Port District, LA
No data for Port ID: 2363 Southern Indiana Maritime District, IN
No data for Port ID: 2367 St. Louis, MO and IL
No data for Port ID: 3204 Toledo-Lucas County Port, OH
No data for Port ID: 3926 Two Harbors, MN
Container shape: (160, 11)


#### Internal Check

In [ ]:
totals={}
for _,row in container.iterrows():
    pid = row['Port_Name']
    if pid not in totals:
        totals[pid] = {'Org':0,'New':0,'EMPTY':0}
    if row['Trade Type'] == 'TOTAL':
        totals[pid]['Org'] = row['Volume']
    else:
        totals[pid]['New'] += row['Volume']
        if row['Trade Type'] == 'EMPTY':
            totals[pid]['EMPTY'] += row['Volume']
        
        

In [ ]:
ngood=0
nbad=0

for port,dct in totals.items():
    org = round(dct['Org'])
    new = round(dct['New'])
    empty = round(dct['EMPTY'])
    if org != new:
        print(f"Port {port}: diff {new - org} Emp {empty}  Org {org} New {new}")   
        nbad+=1
    else:
        ngood+=1
        # if empty == 0:
        #     print(f"EMPTY NOT: Port {port}: Match with EMPTY {empty} Org {org} New {new}")

print(f"Checked {ngood + nbad} ports with {nbad} mismatches and {ngood} good records.") 

#### Container Check vs Socrata

##### EMPTY Check vs Previous Year

In [ ]:
container_hist=df_hist_year_minus_1[df_hist_year_minus_1['Cargo Type'] == 'CONTAINER']

In [ ]:
container_hist_empty = container_hist[container_hist['Trade Type'] == 'EMPTY']  
ntot=0
nfound=0
for _,row in container_hist_empty.iterrows():
    pid = row['Port ID']
    pname = row['Port_Name'].strip()
    vol_hist = row['Volume']        
    hld = container.loc[(container['Port ID'] == pid) & (container['Trade Type'] == 'EMPTY')]
    ntot+=1
    if hld.shape[0] > 0:
        nfound+=1
        volume = hld['Volume'].values[0] 
        if volume > 0 or  vol_hist > 0:
            print(f"Port {pname} ({pid}): HIST EMPTY {vol_hist} vs NEW EMPTY {volume}")
        if volume > 0:
            pp = vol_hist/volume 
  #          print(f"Port {pname} ({pid}): HIST EMPTY {vol_hist} vs NEW EMPTY {volume}  Ratio HIST/NEW {pp:.2f}")
    else:
        print(f"Not in 2022 data: Port {pname} ({pid}): HIST EMPTY {vol_hist} vs NEW EMPTY 0")    
print(f"Checked {ntot} ports with EMPTY data in HIST with {nfound} found in NEW.")

##### Check TOTAL 

In [ ]:
container_socr= pd.read_csv('Container_2022_Ranks_top25_from_Socrata.csv', 
                 sep='\t', 
                 encoding='utf-16',engine='python')


In [ ]:
portXrefsR = {v:k for k,v in portXrefs.items()}
for _,row in container_socr.iterrows():
    port = row['Port'].strip()
    rank = int(row['Rank of TEUs along Port'])
    volume = round(float(row['TEUs'].replace(",","")))
   
    tmp = container.loc[(container['Port_Name'] == port) & (container['Trade Type'] == 'TOTAL')]
    tmpe = container.loc[(container['Port_Name'] == port) & (container['Trade Type'] == 'EMPTY')]
    if tmp.empty:
        if port in portXrefsR:
            xref = portXrefsR[port]
            tmp = container.loc[(container['Port_Name'] == xref) & (container['Trade Type'] == 'TOTAL')]
            tmpe = container.loc[(container['Port_Name'] == xref) & (container['Trade Type'] == 'EMPTY')]
          #  print(f"No match for {port} (xref {xref}) in container data.")
    v = round(tmp['Volume'].values[0]) if not tmp.empty else None
    r = tmp['Port Ranking'].values[0] if not tmp.empty else None
    e = round(tmpe['Volume'].values[0]) if not tmpe.empty else None

    if e is not None: 
       v=v-e # socrata data does not contain info on Empty Containers
    if v != volume or r != rank:
      if v != volume and r != rank:
         wh="B"
      elif v != volume:
         wh="V"
      elif r != rank:
         wh="R"
         nbad+=1
       
      print(f"Bad:  Checking Port ID {port}, Why {wh} New-Rank {r}  Soc-Rank {rank}, New-VOl {v}  Soc-Vol {volume} New-EMP {e} Shape {tmp.shape}  ")
   
   
   
    else:
       ngood+=1

print(f"Checked {nbad + ngood} records with {nbad} mismatches and {ngood} good records.")

#### Check by Type

In [ ]:
a= pd.read_csv('Container_2022_Ranks_top25_byType_from_socrata.csv', 
                 sep='\t', 
                 encoding='utf-16',engine='python')
a.columns = a.columns.str.replace(".1","",regex=False).str.strip()
container_socr = a.T[1:]
container_socr.columns = a.T.iloc[0]
container_socr.reset_index(inplace=True)
container_socr.rename(columns={'Type':'Year',"index":"Port_Name"}, inplace=True    )
container_socr['Year'] = container_socr['Year'].str.strip().astype(int)
container_socr['Domestic, Inbound, Loaded'] = container_socr['Domestic, Inbound, Loaded'].str.replace(",","").astype(int)
container_socr['Domestic, Outbound, Loaded'] = container_socr['Domestic, Outbound, Loaded'].str.replace(",","").astype(int)
container_socr['Foreign, Inbound, Loaded'] = container_socr['Foreign, Inbound, Loaded'].str.replace(",","").astype(int)
container_socr['Foreign, Outbound, Loaded'] = container_socr['Foreign, Outbound, Loaded'].str.replace(",","").astype(int)


In [ ]:
container_socr['IMPORTS'] = container_socr['Domestic, Inbound, Loaded']+ container_socr['Foreign, Inbound, Loaded'] 
container_socr['EXPORTS'] = container_socr['Domestic, Outbound, Loaded']+ container_socr['Foreign, Outbound, Loaded'] 
#container_socr['EMPTY'] = container_socr['Domestic, Inbound, Empty']
container_socr['TOTAL'] = container_socr['IMPORTS'] + container_socr['EXPORTS']  
nbad=0
ngood=0

portXrefsR = {v:k for k,v in portXrefs.items()}
for _,row in container_socr[container_socr['Year'] == REPORTING_YEAR].iterrows():
    port = row['Port_Name'].strip()
    
    for tt in ['IMPORTS','EXPORTS']:
        volume = round(float(row[tt]))
        tmp = container.loc[(container['Port_Name'] == port) & (container['Trade Type'] == tt)]
        if tmp.empty:
            if port in portXrefsR:
                xref = portXrefsR[port]
                tmp = container.loc[(container['Port_Name'] == xref) & (container['Trade Type'] == tt)]
                if tmp.empty:
                   print(f"No match for {port} (xref {xref}) in container data.")
                   continue
        v = round(tmp['Volume'].values[0]) if not tmp.empty else None
    #    r = tmp['Port Ranking'].values[0] if not tmp.empty else None
        if v != volume:
             nbad+=1
             print(f"Bad:  Checking Port ID {port} and Type: {tt},  New-VOl {v}  Soc-Vol {volume}  Shape {tmp.shape}  ")
        else:
             ngood+=1

print(f"Checked {nbad + ngood} records with {nbad} mismatches and {ngood} good records.")

## Combine All Records  

In [36]:
allData = pd.concat([totalTonnage, dryBulk,vesselCalls, top5Com, top5Ag, container], ignore_index=True)
allData.to_csv(f"Port_Performance_{REPORTING_YEAR}_for_Socrata_Joe.csv", index=False)   


In [ ]:
allData.shape

In [ ]:
hist_2021 = df_hist.loc[df_hist['Reporting Year'] == 2021]
hist_2021.shape

In [ ]:
hist_2021.to_csv("Port_Performance_2021_Historical.csv", index=False)

In [ ]:
display(allData['Cargo Type'].value_counts())
display(hist_2021['Cargo Type'].value_counts())

In [ ]:
tp="TOP 5 FOOD/FARM COMMODITIES"
#tp="TOP 5 COMMODITIES"
a = allData.loc[allData['Cargo Type'] == tp]
b = hist_2021.loc[hist_2021['Cargo Type'] == tp]

In [ ]:
c = set(a['Port_Name'].unique().tolist())
d = set(b['Port_Name'].unique().tolist())

print("Ports in 2022 TOTAL TONNAGE not in 2021:", c - d)        
print("Ports in 2021 TOTAL TONNAGE not in 2022:", d - c)

In [ ]:
a = ports_hist['Port_Name'].unique().tolist()
b = hist_2021['Port_Name'].unique().tolist()
c = allData['Port_Name'].unique().tolist()

In [ ]:
print(set(a) - set(b))
print(set(a) - set(c))


### Extra

In [ ]:
Domestic Inbound Loaded
Domestic Inbound Empty
Domestic Outbound Loaded
Domestic Outbound Empty
Foreign Inbound Loaded
Foreign Outbound Loaded